# Notebook 02 v2 — NSL RF retrain (M10 cap fix)

**Why this exists:**

The verification script revealed that NSL v2 RF models were trained WITHOUT the max_depth=20 cap (mean depth 27.8–30.9, max 40–45). UNSW and CIC v2 had the cap applied correctly. This notebook retrains only the 3 NSL RF models with max_depth=20 to align with the other two datasets.

**What this does:**

- Loads NSL v2 preprocessed data (unchanged)
- Retrains rf_binary_cw, rf_5class_smote, rf_5class_cw with max_depth=20
- Overwrites the .pkl files in `models/nsl_kdd_v2/`
- Overwrites the test_pred / test_proba / calib_pred / calib_proba files for these 3 models
- Updates the 3 RF entries in `metrics.json` (preserves the 6 XGB/DNN entries)
- Verifies the max_depth=20 cap actually applied this time

**What this does NOT do:**

- Does NOT touch XGB or DNN models — those were unaffected by M10
- Does NOT modify the model_comparison.csv or imbalance_ablation.csv — those will be regenerated in Day 4 cascade
- Does NOT re-run any downstream notebooks — Day 3 work picks up from these refreshed RF artifacts

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json, pickle, time
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)

PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd_v2'
MODELS_DIR = Path(REPO) / 'models' / 'nsl_kdd_v2'
PREDS_DIR = MODELS_DIR / 'predictions'

X_train = np.load(PROCESSED / 'X_train.npy')
X_calib = np.load(PROCESSED / 'X_calib.npy')
X_test  = np.load(PROCESSED / 'X_test.npy')

y_train_b = np.load(PROCESSED / 'y_train_binary.npy')
y_test_b  = np.load(PROCESSED / 'y_test_binary.npy')

y_train_5 = np.load(PROCESSED / 'y_train_5class.npy')
y_test_5  = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

# Load existing metrics so we preserve XGB and DNN entries
with open(MODELS_DIR / 'metrics.json') as f:
    ALL_METRICS = json.load(f)

print(f'Train: {X_train.shape}  Calib: {X_calib.shape}  Test: {X_test.shape}')
print(f'Existing models in metrics.json: {len(ALL_METRICS)}')
print(f'Names: {list(ALL_METRICS.keys())}')

# Save snapshot of pre-retrain RF metrics for comparison
PRE_METRICS = {name: dict(ALL_METRICS[name]) for name in ALL_METRICS if name.startswith('rf_')}

Train: (100778, 122)  Calib: (25195, 122)  Test: (22544, 122)
Existing models in metrics.json: 9
Names: ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw', 'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote', 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']


## 2. Helpers

In [3]:
RF_MAX_DEPTH = 20  # The cap that was missing from NSL v2 originally

def evaluate(y_true, y_pred, labels):
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0, labels=labels)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
        'f1_per_class': [float(f) for f in f1_per_class],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }

def save_predictions(name, predict_fn):
    test_pred, test_proba = predict_fn(X_test)
    calib_pred, calib_proba = predict_fn(X_calib)
    np.save(PREDS_DIR / f'{name}_test_pred.npy',  test_pred)
    np.save(PREDS_DIR / f'{name}_test_proba.npy', test_proba)
    np.save(PREDS_DIR / f'{name}_calib_pred.npy',  calib_pred)
    np.save(PREDS_DIR / f'{name}_calib_proba.npy', calib_proba)
    return test_pred, test_proba

def report_depths(clf, name, cap):
    depths = [t.tree_.max_depth for t in clf.estimators_]
    mean_d = float(np.mean(depths))
    max_d = int(max(depths))
    cap_ok = max_d <= cap
    flag = '✓' if cap_ok else '✗'
    print(f'  Tree depths: mean={mean_d:.1f}, max={max_d} (cap was {cap}) {flag}')
    return mean_d, max_d

## 3. Retrain rf_binary_cw

In [4]:
name = 'rf_binary_cw'
labels_b = [0, 1]
cw_b = compute_class_weight('balanced', classes=np.array(labels_b), y=y_train_b)
cw_b_dict = {i: cw_b[i] for i in labels_b}

print(f'Retraining {name} with max_depth={RF_MAX_DEPTH}...')
t0 = time.time()
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=RF_MAX_DEPTH,
    n_jobs=-1,
    random_state=SEED,
    class_weight=cw_b_dict,
)
clf.fit(X_train, y_train_b)
print(f'  Trained in {time.time()-t0:.1f}s')

with open(MODELS_DIR / f'{name}.pkl', 'wb') as f:
    pickle.dump(clf, f)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_predictions(name, predict_fn)
metrics = evaluate(y_test_b, test_pred, labels_b)
mean_d, max_d = report_depths(clf, name, RF_MAX_DEPTH)
metrics['max_depth_mean'] = mean_d
metrics['max_depth_actual'] = max_d
ALL_METRICS[name] = metrics

print(f'  acc={metrics["accuracy"]:.4f}  f1m={metrics["f1_macro"]:.4f}  mcc={metrics["mcc"]:.4f}')

# Checkpoint
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2, default=str)

Retraining rf_binary_cw with max_depth=20...
  Trained in 16.4s
  Tree depths: mean=20.0, max=20 (cap was 20) ✓
  acc=0.7768  f1m=0.7760  mcc=0.6166


## 4. Retrain rf_5class_smote

In [5]:
from collections import Counter

# Apply SMOTE (same as original v2 training)
min_class_size = int(np.min(np.bincount(y_train_5)))
k_neighbors = min(5, min_class_size - 1)

smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train_5)

print('Post-SMOTE distribution:')
post = Counter(y_train_sm)
for cid in range(5):
    print(f'  {CLASS_NAMES_5[cid]:8s}: {post[cid]:>7,}')

name = 'rf_5class_smote'
labels_5 = list(range(5))

print(f'\nRetraining {name} with max_depth={RF_MAX_DEPTH}...')
t0 = time.time()
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=RF_MAX_DEPTH,
    n_jobs=-1,
    random_state=SEED,
)
clf.fit(X_train_sm, y_train_sm)
print(f'  Trained in {time.time()-t0:.1f}s')

with open(MODELS_DIR / f'{name}.pkl', 'wb') as f:
    pickle.dump(clf, f)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_predictions(name, predict_fn)
metrics = evaluate(y_test_5, test_pred, labels_5)
mean_d, max_d = report_depths(clf, name, RF_MAX_DEPTH)
metrics['max_depth_mean'] = mean_d
metrics['max_depth_actual'] = max_d
ALL_METRICS[name] = metrics

print(f'  acc={metrics["accuracy"]:.4f}  f1m={metrics["f1_macro"]:.4f}  mcc={metrics["mcc"]:.4f}')
print(f'  Per-class F1: R2L={metrics["f1_per_class"][3]:.3f}  U2R={metrics["f1_per_class"][4]:.3f}')

# Checkpoint
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2, default=str)

Post-SMOTE distribution:
  Normal  :  53,874
  DoS     :  53,874
  Probe   :  53,874
  R2L     :  53,874
  U2R     :  53,874

Retraining rf_5class_smote with max_depth=20...
  Trained in 66.6s
  Tree depths: mean=20.0, max=20 (cap was 20) ✓
  acc=0.7474  f1m=0.5165  mcc=0.6361
  Per-class F1: R2L=0.056  U2R=0.154


## 5. Retrain rf_5class_cw

In [6]:
name = 'rf_5class_cw'
labels_5 = list(range(5))
cw_5 = compute_class_weight('balanced', classes=np.array(labels_5), y=y_train_5)
cw_5_dict = {i: cw_5[i] for i in labels_5}

print(f'Retraining {name} with max_depth={RF_MAX_DEPTH}...')
t0 = time.time()
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=RF_MAX_DEPTH,
    n_jobs=-1,
    random_state=SEED,
    class_weight=cw_5_dict,
)
clf.fit(X_train, y_train_5)
print(f'  Trained in {time.time()-t0:.1f}s')

with open(MODELS_DIR / f'{name}.pkl', 'wb') as f:
    pickle.dump(clf, f)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_predictions(name, predict_fn)
metrics = evaluate(y_test_5, test_pred, labels_5)
mean_d, max_d = report_depths(clf, name, RF_MAX_DEPTH)
metrics['max_depth_mean'] = mean_d
metrics['max_depth_actual'] = max_d
ALL_METRICS[name] = metrics

print(f'  acc={metrics["accuracy"]:.4f}  f1m={metrics["f1_macro"]:.4f}  mcc={metrics["mcc"]:.4f}')
print(f'  Per-class F1: R2L={metrics["f1_per_class"][3]:.3f}  U2R={metrics["f1_per_class"][4]:.3f}')

# Checkpoint
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(ALL_METRICS, f, indent=2, default=str)

Retraining rf_5class_cw with max_depth=20...
  Trained in 15.1s
  Tree depths: mean=20.0, max=20 (cap was 20) ✓
  acc=0.7338  f1m=0.4683  mcc=0.6162
  Per-class F1: R2L=0.006  U2R=0.029


## 6. Before/after comparison

In [7]:
rows = []
for name in ['rf_binary_cw', 'rf_5class_smote', 'rf_5class_cw']:
    pre = PRE_METRICS[name]
    post = ALL_METRICS[name]

    rows.append({
        'Model': name,
        'Pre acc': round(pre['accuracy'], 4),
        'Post acc': round(post['accuracy'], 4),
        'Pre F1m': round(pre['f1_macro'], 4),
        'Post F1m': round(post['f1_macro'], 4),
        'Pre max_d': pre.get('max_depth_actual', '?'),
        'Post max_d': post['max_depth_actual'],
    })
df = pd.DataFrame(rows)
print('NSL RF retrain: before vs after max_depth=20 cap')
print('=' * 90)
print(df.to_string(index=False))
print('=' * 90)

# All max_depth should now be ≤ 20
all_capped = all(ALL_METRICS[n]['max_depth_actual'] <= 20
                 for n in ['rf_binary_cw', 'rf_5class_smote', 'rf_5class_cw'])
if all_capped:
    print('\n✓ All 3 NSL RF models now have max_depth ≤ 20')
    print('  M10 issue resolved across all three datasets (NSL + UNSW + CIC)')
else:
    print('\n✗ Some RF model still exceeds max_depth=20. Investigate.')

NSL RF retrain: before vs after max_depth=20 cap
          Model  Pre acc  Post acc  Pre F1m  Post F1m Pre max_d  Post max_d
   rf_binary_cw   0.7674    0.7768   0.7663    0.7760         ?          20
rf_5class_smote   0.7427    0.7474   0.5102    0.5165         ?          20
   rf_5class_cw   0.7345    0.7338   0.4691    0.4683         ?          20

✓ All 3 NSL RF models now have max_depth ≤ 20
  M10 issue resolved across all three datasets (NSL + UNSW + CIC)


## 7. Commit

In [8]:
os.chdir(REPO)
!git add notebooks/02_nsl_rf_retrain_v2.ipynb
!git status --short
!git commit -m 'Notebook 02-NSL-RF-retrain v2: apply M10 max_depth=20 cap to NSL RF models (was missing)'
!git push origin main

Refresh index: 100% (234/234), done.
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_cic_train_models_v2.ipynb
AM notebooks/02_nsl_rf_retrain_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
?? calibrators/
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[main 558a998] Notebook 02-NSL-RF-retrain v2: apply M10 max_depth=20 cap to NSL RF models (was missing)
 1 file changed, 1 insertion(+)
 create mode 100644 notebooks/02_nsl_rf_retrain_v2.ipynb
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 4.53 KiB | 662.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects

In [9]:
import os
os.chdir('/content/drive/MyDrive/XIDS_Research/xids-research')
!git add docs/v2_rebuild_progress_day1_day2.md
!git commit -m "Progress doc v8: final round audit"
!git push origin main

[main 9fb0ba0] Progress doc v8: final round audit
 1 file changed, 458 insertions(+)
 create mode 100644 docs/v2_rebuild_progress_day1_day2.md
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 9.98 KiB | 1021.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/anasbiswas1/xids-research.git
   558a998..9fb0ba0  main -> main
